# Threads, processes, and the Python GIL

**Core notebook, about 25 minutes.** Predict each result before running it. The goal is to choose threads or processes based on the work, not to memorize one choice as always faster.

## Mental model

```text
one process                         several processes
+-----------------------------+    +-----------+  +-----------+
| one interpreter and one GIL |    | interpreter|  | interpreter|
| thread  thread  thread      |    | and GIL    |  | and GIL    |
| shared memory               |    | own memory |  | own memory |
+-----------------------------+    +-----------+  +-----------+
```

- Threads share one process and its memory.
- The Global Interpreter Lock, or GIL, is a CPython rule that lets only one thread run Python code at a time.
- Processes can run Python code on separate CPU cores, but starting them and sending data takes time and memory.
- NumPy and compiled Numba functions can let threads run at the same time during some calculations.

In [ ]:
import os
import time
import dask
from dask import delayed
from workloads import fib, sleepy

# Use four workers so the result is easy to understand and resources are not wasted.
n_workers = min(4, os.cpu_count() or 1)
print("Available logical CPUs:", os.cpu_count())
print("Workers used in this notebook:", n_workers)

## 1. Work that spends its time calculating in Python

**Predict first:** which will be faster for several recursive Fibonacci calculations, threads or processes? What cost could make the result less clear for a very small task?

In [ ]:
cpu_tasks = [delayed(fib)(32) for _ in range(n_workers)]

started = time.perf_counter()
thread_results = dask.compute(
    *cpu_tasks, scheduler="threads", num_workers=n_workers
)
thread_time = time.perf_counter() - started

started = time.perf_counter()
process_results = dask.compute(
    *cpu_tasks, scheduler="processes", num_workers=n_workers
)
process_time = time.perf_counter() - started

assert thread_results == process_results
print(f"Threads:   {thread_time:.3f} s")
print(f"Processes: {process_time:.3f} s")

<details><summary>Interpretation</summary>

The GIL lets only one thread run this Python calculation at a time. Separate processes can run on several CPU cores. For very small tasks, the time needed to start processes and send them data can remove that advantage.

</details>

## 2. Waiting work

`time.sleep` represents waiting for a file or network response and lets other threads run. **Predict first:** which choice should finish sooner here?

In [ ]:
waiting_tasks = [delayed(sleepy)(0.5) for _ in range(n_workers)]

started = time.perf_counter()
thread_wait_results = dask.compute(
    *waiting_tasks, scheduler="threads", num_workers=n_workers
)
thread_wait_time = time.perf_counter() - started

started = time.perf_counter()
process_wait_results = dask.compute(
    *waiting_tasks, scheduler="processes", num_workers=n_workers
)
process_wait_time = time.perf_counter() - started

assert thread_wait_results == process_wait_results
print(f"Threads:   {thread_wait_time:.3f} s")
print(f"Processes: {process_wait_time:.3f} s")

## Your turn: explain the evidence

**12 minutes.** Discuss with a neighbor:

1. Did both results match your predictions?
2. What would happen if every task received a 2 GB array?
3. Which choice would you try for a task that spends most of its time in NumPy?
4. What measurement would you collect before changing a real program?

Blue sticky note means you need help. Yellow means you can explain your choice.

## Takeaway

- Try threads for work that mostly waits, or for NumPy and Numba calculations that let threads run together.
- Try processes for larger tasks that spend most of their time calculating in Python.
- Limit the worker count. Too many workers can slow each other down and use too much memory.
- Time your real program because task size, libraries, and data movement affect the result.

Next: [`../5_dask/1_delayed.ipynb`](../5_dask/1_delayed.ipynb).